# CMPE 249 detection: Stage 2 runner
Runtime → Change runtime type → **L4 GPU** (use the same GPU type for every latency number).

No Google Drive: data lives on Colab local disk, and results are pushed to this repo's `results` branch after every run.
One-time setup: add a GitHub token as a Colab Secret named `GITHUB_TOKEN` (key icon in the left sidebar), with
*Contents: read and write* on `shloakk/2d-object-detection` only.

Every cell is safe to re-run: clones are pulled instead of re-cloned, and data already on the
runtime disk is not downloaded again.

In [ ]:
# Which runtime is this? (GPU model, TPU or CPU)
import glob, os, shutil, subprocess
SHORT_NAMES = {'L4': 'L4', 'A100': 'A100', 'H100': 'H100', 'T4': 'T4', 'RTX PRO 6000': 'G4'}
gpu = ''
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
if gpu:
    short = next((v for k, v in SHORT_NAMES.items() if k in gpu), 'other GPU')
    print(f'Runtime: {short} GPU  ({gpu})')
elif glob.glob('/dev/accel*') or glob.glob('/dev/vfio/[0-9]*') or os.environ.get('TPU_ACCELERATOR_TYPE'):
    print(f"Runtime: TPU ({os.environ.get('TPU_ACCELERATOR_TYPE', 'type not reported')}): these PyTorch models need a GPU")
else:
    print('Runtime: CPU only: switch to a GPU runtime')
if gpu and short != 'L4':
    print('Note: every Stage 2 latency number should come from the same GPU type (L4 planned).')

In [ ]:
import os
REPO = '/content/2d-object-detection'
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/shloakk/2d-object-detection.git {REPO}
%cd {REPO}
!git log -1 --oneline
!pip install -q -r requirements.txt

In [ ]:
import os
# Results go into a clone of the `results` branch; src.sync_results commits and pushes it.
if os.path.isdir('/content/results/.git'):
    !git -C /content/results pull -q --rebase
else:
    from getpass import getpass
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = getpass('GitHub token: ')
    !git clone -q -b results https://{token}@github.com/shloakk/2d-object-detection.git /content/results
    !git -C /content/results config user.name 'Colab runner'
    !git -C /content/results config user.email 'colab@users.noreply.github.com'
os.environ['CMPE249_DATA'] = '/content/data'        # Colab local disk: fast reads
os.environ['CMPE249_WEIGHTS'] = '/content/weights'
os.environ['CMPE249_RESULTS'] = '/content/results'
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no CUDA GPU')

In [ ]:
# Data prep. Safe to re-run: each script checks what is already on the runtime disk and only
# downloads what is missing (the disk is wiped whenever the runtime resets).
!python -m src.data.kitti_to_coco            # full KITTI val (1496 images)
!python -m src.data.coco_subset --full       # full COCO val2017 (5000 images, fetched by coco_url)

In [ ]:
# One config end to end, then push its results to the `results` branch
!python -m src.predict --config configs/full/yolo26s_kitti.yaml && python -m src.sync_results

In [ ]:
# All accuracy runs (Faster R-CNN COCO is the slowest). Each run is pushed as soon as it finishes.
for cfg in ['yolo26s_coco', 'frcnn_v2_coco', 'rtdetr_l_coco', 'yolo26s_o2m_coco',
            'rtdetr_l_kitti', 'frcnn_v2_kitti', 'yolo26s_o2m_kitti', 'yolo26s_1280_kitti']:
    !python -m src.predict --config configs/full/{cfg}.yaml && python -m src.sync_results

In [ ]:
# Latency: batch 1, 50 warm-up + 200 timed, FP32 and FP16, on 200 KITTI images
for m in ['yolo26s', 'yolo26s_o2m', 'yolo26s_1280', 'rtdetr_l', 'frcnn_v2']:
    for half in ['', '--half']:
        !python -m src.bench_latency --config configs/full/{m}_kitti.yaml {half}
!python -m src.sync_results